In [3]:
import os
import json
import numpy as np
from scipy.stats import t, norm

In [4]:
def calcular_kpis_promedio_con_ic_en_formato_json(directorio, nivel_confianza=95, guardar_en=None):
    """
    Calcula el promedio ± IC de todos los KPIs numéricos en archivos JSON de un directorio.
    Usa t-Student si n < 100, o normal estándar (z) si n >= 100.

    Args:
        directorio (str): Ruta a carpeta con archivos JSON.
        nivel_confianza (int or float): Nivel de confianza deseado (ej. 90, 95, 99).
        guardar_en (str or None): Ruta para guardar el resultado en un JSON (opcional).

    Returns:
        dict: Diccionario anidado con 'media ± error' y metadata de cálculo.
    """
    if not (50 <= nivel_confianza < 100):
        raise ValueError("El nivel de confianza debe estar entre 50 y 99.9")

    prob = nivel_confianza / 100
    json_files = [os.path.join(directorio, f) for f in os.listdir(directorio) if f.endswith(".json")]
    n = len(json_files)
    if n == 0:
        raise ValueError("No se encontraron archivos JSON en el directorio.")

    # Usar el primer archivo como referencia de estructura
    with open(json_files[0], "r") as f:
        ejemplo = json.load(f)

    def extraer_rutas(d, prefijo=""):
        rutas = []
        for k, v in d.items():
            ruta = f"{prefijo}.{k}" if prefijo else k
            if isinstance(v, dict):
                rutas += extraer_rutas(v, ruta)
            elif isinstance(v, (int, float)):
                rutas.append(ruta)
        return rutas

    kpis_ruta = extraer_rutas(ejemplo)

    # Recolectar valores de cada KPI
    data = {kpi: [] for kpi in kpis_ruta}
    for file in json_files:
        with open(file, "r") as f:
            contenido = json.load(f)
            for kpi in kpis_ruta:
                try:
                    val = contenido
                    for key in kpi.split("."):
                        val = val[key]
                    if isinstance(val, (int, float)):
                        data[kpi].append(val)
                except (KeyError, TypeError):
                    continue

    resumen = {"_info": {"n": n, "nivel_confianza": f"{nivel_confianza}%", "distribucion": ""}}

    if n > 100:
        z_val = norm.ppf((1 + prob) / 2)
        resumen["_info"]["distribucion"] = "normal"
    else:
        resumen["_info"]["distribucion"] = "t_student"

    for kpi, valores in data.items():
        arr = np.array(valores)
        mean = np.mean(arr)
        std = np.std(arr, ddof=1)
        sem = std / np.sqrt(n)
        if n > 100:
            error = z_val * sem
        else:
            t_val = t.ppf((1 + prob) / 2, df=n - 1)
            error = t_val * sem
        valor_str = f"{round(mean, 2)} ± {round(error, 2)}"

        puntero = resumen
        keys = kpi.split(".")
        for key in keys[:-1]:
            puntero = puntero.setdefault(key, {})
        puntero[keys[-1]] = valor_str

    if guardar_en:
        with open(guardar_en, "w") as f:
            json.dump(resumen, f, indent=4)

    return resumen

In [18]:
resumen = calcular_kpis_promedio_con_ic_en_formato_json("resultados simulacion/ModeloProactivo_T4500_C4208/kpis", nivel_confianza=99, guardar_en=None)
display(resumen)

{'_info': {'n': 10, 'nivel_confianza': '99%', 'distribucion': 't_student'},
 'LOS_hospitalizado': {'por_hospital_y_unidad': {'Hospital_1': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '2.3 ± 0.14', 'tratamiento': '67.27 ± 0.84'},
    'OR': {'espera': '0.72 ± 0.04', 'tratamiento': '13.04 ± 0.04'},
    'SDU_WARD': {'espera': '0.01 ± 0.01', 'tratamiento': '199.49 ± 0.86'}},
   'Hospital_2': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '2.0 ± 0.13', 'tratamiento': '67.85 ± 0.9'},
    'OR': {'espera': '0.67 ± 0.05', 'tratamiento': '13.06 ± 0.04'},
    'SDU_WARD': {'espera': '0.01 ± 0.0', 'tratamiento': '197.11 ± 1.25'}},
   'Hospital_3': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '1.33 ± 0.07', 'tratamiento': '64.7 ± 0.43'},
    'OR': {'espera': '0.41 ± 0.03', 'tratamiento': '13.02 ± 0.04'},
    'SDU_WARD': {'espera': '0.0 ± 0.0', 'tratamiento': '186.38 ± 1.01'}}},
  'promedio_por_hospital': {'Hospital_1': '260.61 ± 1.04',
   'Ho

In [17]:
resumen = calcular_kpis_promedio_con_ic_en_formato_json("resultados simulacion/ModeloProactivoExBueno_T4500_C4208/kpis", nivel_confianza=99, guardar_en=None)
display(resumen)

{'_info': {'n': 100, 'nivel_confianza': '99%', 'distribucion': 't_student'},
 'LOS_hospitalizado': {'por_hospital_y_unidad': {'Hospital_1': {'ED': '0.0 ± 0.0',
    'GA': '39.1 ± 0.37',
    'ICU': {'espera': '2.66 ± 0.03', 'tratamiento': '67.4 ± 0.15'},
    'OR': {'espera': '0.95 ± 0.01', 'tratamiento': '12.72 ± 0.01'},
    'SDU_WARD': {'espera': '0.07 ± 0.0', 'tratamiento': '198.1 ± 0.22'}},
   'Hospital_2': {'ED': '0.0 ± 0.0',
    'GA': '23.32 ± 0.26',
    'ICU': {'espera': '3.07 ± 0.03', 'tratamiento': '62.38 ± 0.15'},
    'OR': {'espera': '0.99 ± 0.01', 'tratamiento': '12.68 ± 0.01'},
    'SDU_WARD': {'espera': '0.04 ± 0.0', 'tratamiento': '189.97 ± 0.19'}},
   'Hospital_3': {'ED': '0.0 ± 0.0',
    'GA': '18.47 ± 0.3',
    'ICU': {'espera': '2.11 ± 0.04', 'tratamiento': '62.53 ± 0.15'},
    'OR': {'espera': '0.53 ± 0.01', 'tratamiento': '12.98 ± 0.01'},
    'SDU_WARD': {'espera': '0.03 ± 0.0', 'tratamiento': '184.72 ± 0.25'}}},
  'promedio_por_hospital': {'Hospital_1': '290.3 ± 0.43

In [32]:
costo_promedio = resumen["costo_diario_promedio"]["General"]["total"]
costo_promedio = float(costo_promedio.split(" ± ")[0]) + float(costo_promedio.split(" ± ")[1])
costo_promedio

1279.25

In [ ]:
"""
"costo_diario_promedio": {"General": {
"social_en_wl": 5952.24,
"social_en_hospitales": 12766.53,
"social": 18718.76,
"derivaciones_wl": 2477.6,
"derivaciones_ed": 697.25,
"traslados": 116.67,
"operativo": 3291.53,
"total": 22010.29},

'costo_diario_promedio': {'General': {'social_en_wl': '77.26 ± 14.26',
'social_en_hospitales': '122.14 ± 7.77',
'social': '199.4 ± 20.02',
'derivaciones_wl': '25.94 ± 4.09',
'derivaciones_ed': '842.45 ± 64.65',
'traslados': '93.73 ± 2.72',
'operativo': '962.13 ± 67.23',
'total': '1161.53 ± 81.86'},
"""

'\n"costo_diario_promedio": {"General": {\n"social_en_wl": 5952.24,\n"social_en_hospitales": 12766.53,\n"social": 18718.76,\n"derivaciones_wl": 2477.6,\n"derivaciones_ed": 697.25,\n"traslados": 116.67,\n"operativo": 3291.53,\n"total": 22010.29},\n\n\'costo_diario_promedio\': {\'General\': {\'social_en_wl\': \'97.61 ± 27.75\',\n\'social_en_hospitales\': \'113.98 ± 6.73\',\n\'social\': \'211.58 ± 31.32\',\n\'derivaciones_wl\': \'501.65 ± 4.93\',\n\'derivaciones_ed\': \'709.46 ± 58.89\',\n\'traslados\': \'91.98 ± 2.48\',\n\'operativo\': \'1303.08 ± 62.71\',\n\'total\': \'1514.66 ± 76.16\'},\n\n\'costo_diario_promedio\': {\'General\': {\'social_en_wl\': \'74.73 ± 17.48\',\n\'social_en_hospitales\': \'123.69 ± 9.62\',\n\'social\': \'198.41 ± 25.57\',\n\'derivaciones_wl\': \'0.0 ± 0.0\',\n\'derivaciones_ed\': \'859.89 ± 70.93\',\n\'traslados\': \'93.4 ± 2.26\',\n\'operativo\': \'953.29 ± 71.77\',\n\'total\': \'1151.7 ± 91.3\'},\n\n\'costo_diario_promedio\': {\'General\': {\'social_en_wl\': \